# UrbanSense AI — YOLO11n Road Damage Detection (Google Colab)

This notebook fine-tunes **YOLO11n** on the **RDD2022 India** road damage dataset with Google Drive checkpoint persistence.

### Target 4-Class Taxonomy:
- **Class 0 (D00)**: Longitudinal Crack (wheel path fatigue)
- **Class 1 (D10)**: Transverse Crack (cross-road thermal/reflective crack)
- **Class 2 (D20)**: Alligator Crack (fatigue mesh cracking)
- **Class 3 (D40)**: Pothole (road surface cavity)

> **Orientation Guard**: Crack orientation defines distress classification. Rotation (`degrees=0.0`) and vertical flips (`flipud=0.0`) are disabled to ensure ground-truth distinction between longitudinal (D00) and transverse (D10) cracks.

## 1. Hardware & GPU Diagnostic

In [ ]:
import torch
import os

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device:      {device_name}')
    print(f'Total VRAM:      {total_vram:.2f} GB')
else:
    print('❌ ERROR: No GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.')


## 2. Mount Google Drive (Recommended for Checkpoint Persistence)

Mounting Google Drive saves model checkpoints (`last.pt`, `best.pt`) and training logs to Google Drive so your progress is preserved even if the Colab session disconnects.

In [ ]:
from google.colab import drive
from pathlib import Path

try:
    drive.mount('/content/drive')
    drive_mounted = True
    gdrive_project = Path('/content/drive/MyDrive/UrbanSense_AI/runs')
    gdrive_project.mkdir(parents=True, exist_ok=True)
    print(f'✅ Google Drive mounted. Checkpoints will sync to: {gdrive_project}')
except Exception as err:
    drive_mounted = False
    print(f'⚠️ Google Drive not mounted ({err}). Checkpoints will be saved locally in /content/runs.')


## 3. Install Ultralytics & Dependencies

In [ ]:
!pip install -q --upgrade 'ultralytics>=8.3.0' opencv-python matplotlib pyyaml

import ultralytics
ultralytics.checks()


## 4. Dataset Setup & Colab Portability

Locates or unpacks the prepared RDD2022 dataset in `/content/datasets/rdd2022` and creates a dynamic `dataset_colab.yaml` specification.

In [ ]:
import yaml
from pathlib import Path

# Check common Colab dataset paths
candidates = [
    Path('/content/drive/MyDrive/UrbanSense_AI/datasets/rdd2022'),
    Path('/content/UrbanSense-AI/datasets/rdd2022'),
    Path('/content/datasets/rdd2022'),
    Path('./datasets/rdd2022'),
]

dataset_root = None
for c in candidates:
    if (c / 'images' / 'train').is_dir() and (c / 'labels' / 'train').is_dir():
        dataset_root = c.resolve()
        break

if dataset_root is None:
    print('⚠️ Dataset not found in standard paths.')
    print('Please upload your rdd2022 zip archive or git clone the project:')
    print('  !git clone https://github.com/<your-repo>/UrbanSense-AI.git')
    dataset_root = Path('/content/datasets/rdd2022').resolve()
else:
    print(f'✅ Found prepared RDD2022 dataset at: {dataset_root}')

# Generate runtime dataset YAML for Colab
colab_yaml_path = Path('/content/dataset_colab.yaml')
yaml_content = {
    'path': str(dataset_root),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'longitudinal_crack',
        1: 'transverse_crack',
        2: 'alligator_crack',
        3: 'pothole',
    },
}
with open(colab_yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f'Generated Colab dataset specification at: {colab_yaml_path}')
print(yaml_content)


## 5. Dataset Audit & Label Health Check

In [ ]:
from collections import Counter

train_img_dir = dataset_root / 'images' / 'train'
val_img_dir = dataset_root / 'images' / 'val'
train_lbl_dir = dataset_root / 'labels' / 'train'
val_lbl_dir = dataset_root / 'labels' / 'val'

train_imgs = list(train_img_dir.glob('*.jpg')) if train_img_dir.is_dir() else []
val_imgs = list(val_img_dir.glob('*.jpg')) if val_img_dir.is_dir() else []
print(f'Training images:   {len(train_imgs)}')
print(f'Validation images: {len(val_imgs)}')

def count_labels(lbl_dir):
    counts = Counter()
    empty = 0
    for p in lbl_dir.glob('*.txt'):
        text = p.read_text().strip()
        if not text:
            empty += 1
            continue
        for line in text.splitlines():
            parts = line.strip().split()
            if parts:
                counts[int(parts[0])] += 1
    return counts, empty

if train_lbl_dir.is_dir():
    t_counts, t_empty = count_labels(train_lbl_dir)
    v_counts, v_empty = count_labels(val_lbl_dir)
    print(f'Negative background samples: Train={t_empty}, Val={v_empty}')
    names = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
    for i, name in enumerate(names):
        print(f'  Class {i} ({name}): Train={t_counts[i]}, Val={v_counts[i]}')


## 6. Initialize Pretrained YOLO11n

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
print('Model architecture initialized: YOLO11n')


## 7. Model Training Loop (50 Epochs, AdamW, Orientation Guards)

### Configuration:
- **Epochs**: `50`
- **Batch Size**: `8`
- **Optimizer**: `AdamW` (`lr0=0.001`)
- **Early Stopping**: `patience=10`
- **Workers**: `2`
- **Seed**: `42`
- **Orientation Augmentations**: `degrees=0.0`, `flipud=0.0`, `fliplr=0.5`, `mosaic=1.0`

In [ ]:
local_project = Path('/content/runs/train_road_damage')
project_dir = str(gdrive_project / 'train_road_damage') if drive_mounted else str(local_project)
run_name = 'yolo11n_rdd2022_india'

print(f'Project output directory: {project_dir}/{run_name}')

results = model.train(
    data=str(colab_yaml_path),
    epochs=50,
    patience=10,
    batch=8,
    imgsz=640,
    device=0,
    workers=2,
    seed=42,
    project=project_dir,
    name=run_name,
    optimizer='AdamW',
    lr0=0.001,
    save=True,
    save_period=5,
    val=True,
    plots=True,
    verbose=True,
    # Orientation-safe augmentations
    degrees=0.0,
    fliplr=0.5,
    flipud=0.0,
    mixup=0.0,
    mosaic=1.0,
)


## 8. Checkpoint Resume Cell (Disconnection Recovery)

If the Colab session disconnects, run this cell to resume training from the latest checkpoint stored in Google Drive or local storage.

In [ ]:
# To resume training from checkpoint:
# last_pt = Path(project_dir) / run_name / 'weights' / 'last.pt'
# if last_pt.is_file():
#     print(f'Resuming from checkpoint: {last_pt}')
#     resume_model = YOLO(str(last_pt))
#     resume_model.train(resume=True)
# else:
#     print('No checkpoint found to resume.')


## 9. Validation Evaluation & Per-Class Metrics

In [ ]:
best_weights = Path(project_dir) / run_name / 'weights' / 'best.pt'
if best_weights.is_file():
    print(f'Evaluating best model: {best_weights}')
    eval_model = YOLO(str(best_weights))
    metrics = eval_model.val(data=str(colab_yaml_path), split='val')
    print(f'\nOverall mAP50:    {metrics.box.map50:.4f}')
    print(f'Overall mAP50-95: {metrics.box.map:.4f}')
    print(f'Overall Precision: {metrics.box.mp:.4f}')
    print(f'Overall Recall:    {metrics.box.mr:.4f}')
    print('\nPer-Class Performance:')
    names = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
    for i, name in enumerate(names):
        p = metrics.box.p[i] if i < len(metrics.box.p) else 0.0
        r = metrics.box.r[i] if i < len(metrics.box.r) else 0.0
        ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0.0
        print(f'  {name:<20} | Precision: {p:.3f} | Recall: {r:.3f} | mAP50: {ap50:.3f}')
else:
    print(f'Weights not found at {best_weights}. Complete training first.')


## 10. Visualizing Training Loss & Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import cv2

run_dir = Path(project_dir) / run_name
plots = ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']

for plot_name in plots:
    plot_path = run_dir / plot_name
    if plot_path.is_file():
        img = cv2.imread(str(plot_path))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(12, 6))
            plt.imshow(img_rgb)
            plt.axis('off')
            plt.title(plot_name)
            plt.show()


## 11. Model Download & UrbanSense AI Deployment

### Deployment Target:
Download `best.pt` and place it in your local workspace at:
```
ai_engine/models/road_damage_yolo11n.pt
```

In [ ]:
from google.colab import files

best_weights = Path(project_dir) / run_name / 'weights' / 'best.pt'
if best_weights.is_file():
    print(f'Triggering download for best.pt ({best_weights.stat().st_size / (1024*1024):.2f} MB)...')
    files.download(str(best_weights))
    print('Target location: ai_engine/models/road_damage_yolo11n.pt')
else:
    print('best.pt not yet generated.')
